## Setup

Requires `OPENAI_API_KEY` in a `.env` file (and `OPENAI_BASE_URL` if you use a proxy such as Vocareum).
The databases (`data/external/cultpass.db` and `data/core/udahub.db`) are included, so you can run this notebook straight away.
Re-run `01_external_db_setup.ipynb` and `02_core_db_setup.ipynb` only to reset the data: the scenarios below make real bookings and cancellations and store tickets and memories.

In [1]:
import logging
from dotenv import load_dotenv
from utils import chat_interface

In [2]:
load_dotenv()

# Each scenario's decision trace is printed by `show()` below. To also stream the agents' decisions
# live (useful in the interactive chat), uncomment:
# logging.basicConfig(format="%(message)s")
# logging.getLogger("udahub").setLevel(logging.INFO)

True

## Run

In [3]:
# IDEALLY YOUR ONLY IMPORT HERE IS:
# from agentic.workflow import orchestrator

from agentic.workflow import orchestrator

**Architecture** (see `agentic/design/README.md`)

In [4]:
print(orchestrator.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	load_context(load_context)
	classifier(classifier)
	supervisor(supervisor)
	knowledge_agent(knowledge_agent)
	account_agent(account_agent)
	quality_gate(quality_gate)
	approve(approve)
	escalation_agent(escalation_agent)
	request_info(request_info)
	finalize(finalize)
	write_memory(write_memory)
	__end__([<p>__end__</p>]):::last
	__start__ --> load_context;
	account_agent --> quality_gate;
	approve --> finalize;
	classifier --> supervisor;
	escalation_agent --> finalize;
	finalize --> write_memory;
	knowledge_agent --> quality_gate;
	load_context --> classifier;
	quality_gate -.-> approve;
	quality_gate -.-> escalation_agent;
	request_info --> finalize;
	supervisor -.-> account_agent;
	supervisor -.-> escalation_agent;
	supervisor -.-> knowledge_agent;
	supervisor -.-> request_info;
	write_memory --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	cla

## Scenarios

Each scenario uses its own ticket id (= LangGraph `thread_id`). `process_ticket` sends a message plus optional ticket metadata;
`show` prints the decision trace and the reply.

In [5]:
from agentic.workflow import process_ticket


def show(state):
    for step in state["trace"]:
        print("  ", step)
    print("\nAssistant:", state["final_response"], "\n")

**1. Knowledge question from an unknown customer → RAG resolver**

In [6]:
show(process_ticket("Hi! What's included in a CultPass subscription? Are premium events extra?", "demo-1"))

   [memory] ticket=demo-1 customer=Guest identified=False status=open
   [classifier] general_info urgency=low sentiment=neutral account_action=False confidence=0.90
   [supervisor] route -> knowledge_agent: general_info can be answered from the knowledge base
   [knowledge_agent] retrieved ["What's Included in a CultPass Subscription (0.775)", 'Premium Experiences and Subscription Tiers (0.622)', 'How to Reserve a Spot for an Event (0.534)']
   [quality_gate] confidence=0.90 grounded=True needs_human=False: The draft accurately summarizes the contents of a CultPass subscription and mentions the potential additional fees for premium experiences, aligning with the knowledge base.
   [quality_gate] approved draft
   [finalize] status=resolved
   [memory] guest customer: nothing stored

Assistant: Hello! Your CultPass subscription includes 4 curated experiences each month. You can choose from a variety of options like art exhibitions, museum entries, music concerts, film screenings, and m

**2. Account action for a known customer → account agent with tools**

Cathy (basic tier) asks about her reservations, then cancels one in the same session (short-term memory).

In [7]:
cathy = {"external_user_id": "88382b", "channel": "chat"}
show(process_ticket("Which reservations do I have coming up?", "demo-2", cathy))

   [memory] ticket=demo-2 customer=Cathy Bloom identified=True status=open
   [classifier] reservation urgency=medium sentiment=neutral account_action=True confidence=0.90
   [supervisor] route -> account_agent: reservation needs account data/actions
   [account_agent] tools used: ['get_my_reservations']
   [quality_gate] confidence=0.90 grounded=True needs_human=False: The draft accurately lists the upcoming reservation details based on the tool results and offers further assistance.
   [quality_gate] approved draft
   [finalize] status=resolved
   [memory] stored 0 new memories: []

Assistant: You have one upcoming reservation:

- **Experience:** Sunset Paddleboarding
- **Location:** Santa Catarina, Brazil
- **Date:** October 18, 2026
- **Status:** Reserved

If you need any further assistance or want to make changes, just let me know! 



In [8]:
show(process_ticket("Please cancel that one, I can't make it.", "demo-2"))

   [memory] ticket=demo-2 customer=Cathy Bloom identified=True status=resolved
   [classifier] reservation urgency=medium sentiment=neutral account_action=True confidence=0.90
   [supervisor] route -> account_agent: reservation needs account data/actions
   [account_agent] tools used: ['get_my_reservations', 'get_my_account', 'cancel_my_reservation']
   [quality_gate] confidence=0.90 grounded=True needs_human=False: The draft accurately reflects the successful cancellation of the reservation and the return of credit, based on the tool results.
   [quality_gate] approved draft
   [finalize] status=resolved
   [memory] stored 0 new memories: []

Assistant: Your reservation for **Sunset Paddleboarding** on **October 18, 2026**, has been successfully canceled. You will receive a credit for this cancellation.

If you need anything else, feel free to ask! 



**3. Guest needs an account action → asks for identification, then continues on the same thread**

In [9]:
show(process_ticket("I'd like to book the Samba Night at Lapa.", "demo-3"))

   [memory] ticket=demo-3 customer=Guest identified=False status=open
   [classifier] reservation urgency=medium sentiment=neutral account_action=True confidence=0.90
   [supervisor] route -> request_info: account action needed but customer is not identified
   [request_info] asked customer to identify themselves
   [finalize] status=awaiting_customer
   [memory] guest customer: nothing stored

Assistant: I'd be happy to help with that! To look into your account, could you please share the email address registered with your CultPass account? 



In [10]:
show(process_ticket("Sure, it's bob.stone@granite.com", "demo-3"))

   [memory] ticket=demo-3 customer=Guest identified=False status=awaiting_customer
   [classifier] reservation urgency=medium sentiment=neutral account_action=True confidence=0.90
   [supervisor] identified customer Bob Stone via bob.stone@granite.com
   [supervisor] route -> account_agent: reservation needs account data/actions
   [account_agent] tools used: ['find_experiences', 'book_experience', 'get_my_reservations']
   [quality_gate] confidence=0.90 grounded=True needs_human=False: The draft correctly informs the customer about their existing reservation for the Samba Night at Lapa and offers further assistance if needed.
   [quality_gate] approved draft
   [finalize] status=resolved
   [memory] stored 0 new memories: []

Assistant: It looks like you already have a reservation for the "Samba Night at Lapa" on October 20, 2026. If you need any changes or have other requests, just let me know! 



**4. Blocked account → escalated to Trust & Safety without revealing the reason**

In [11]:
show(process_ticket("I can't log into my account, it keeps failing!", "demo-4", {"email": "alice.kingsley@wonderland.com"}))

   [memory] ticket=demo-4 customer=Alice Kingsley identified=True status=open
   [classifier] account_blocked urgency=high sentiment=frustrated account_action=False confidence=0.90
   [supervisor] route -> escalation_agent: account is blocked: policy requires Trust & Safety review
   [escalation_agent] -> trust_and_safety (high): account is blocked: policy requires Trust & Safety review
   [finalize] status=escalated
   [memory] stored 1 new memories: ['Reported an issue with logging into their account, stating that it keeps failing.']

Assistant: Dear Alice,

Thank you for reaching out to us. I understand that you're having trouble logging into your account, and I apologize for the inconvenience this has caused. Your account is currently under review by our Trust & Safety team, and a specialist will follow up with you shortly. Please expect a response via email within one business day.

If you have any further questions in the meantime, feel free to reach out.

Best regards,
CultPass 

**5. Refund request → registered via the internal refund tool and escalated for approval (policy)**

In [12]:
show(process_ticket("I was charged twice for my subscription this month. I want my money back.", "demo-5",
                    {"external_user_id": "e6376d", "channel": "email", "urgency": "high"}))

   [memory] ticket=demo-5 customer=Frank Ocean identified=True status=open
   [classifier] billing_refund urgency=high sentiment=frustrated account_action=True confidence=0.90
   [supervisor] route -> account_agent: billing_refund needs account data/actions
   [account_agent] tools used: ['request_refund']
   [quality_gate] confidence=0.90 grounded=True needs_human=True: The draft correctly informs the customer about the refund request status but cannot finalize the refund without human approval.
   [escalation_agent] -> billing (high): policy requires human approval for ['request_refund']
   [finalize] status=escalated
   [memory] stored 1 new memories: ['Reported being charged twice for their subscription and requested a refund.']

Assistant: Dear Frank,

Thank you for reaching out regarding the double charge on your subscription. I've registered your refund request, and it is currently pending approval from our support lead. Please note that I can't promise a specific amount or time

**6. Long-term memory: Bob had a login ticket 3 days ago, and we know his preferences**

The agents recall his previous ticket (password reset already tried) and his stored preferences.

In [13]:
bob = {"email": "bob.stone@granite.com"}
show(process_ticket("I still can't log in, the password reset email never arrives.", "demo-6", bob))

   [memory] ticket=demo-6 customer=Bob Stone identified=True status=open
   [classifier] login_access urgency=high sentiment=frustrated account_action=False confidence=0.90
   [supervisor] route -> escalation_agent: repeat login_access issue: the customer reported it before
   [escalation_agent] -> technical_support (high): repeat login_access issue: the customer reported it before
   [finalize] status=escalated
   [memory] stored 1 new memories: ['Reported an issue with not receiving the password reset email.']

Assistant: Hi Bob,

Thank you for reaching out again. I understand how frustrating it can be to not receive the password reset email. I've escalated your issue to our technical support team, and they will follow up with you shortly. Please expect a response via email within one business day.

If you have any other questions in the meantime, feel free to ask!

Best regards,
CultPass Support Team 



In [14]:
show(process_ticket("Can you recommend an experience for me to book this week?", "demo-7", bob))

   [memory] ticket=demo-7 customer=Bob Stone identified=True status=open
   [classifier] reservation urgency=medium sentiment=neutral account_action=True confidence=0.90
   [supervisor] route -> account_agent: reservation needs account data/actions
   [account_agent] tools used: ['find_experiences']
   [quality_gate] confidence=0.90 grounded=True needs_human=False: The draft provides accurate information about available experiences and suggests a booking, while also asking a clarifying question to proceed.
   [quality_gate] approved draft
   [finalize] status=awaiting_customer
   [memory] stored 0 new memories: []

Assistant: I found a couple of great experiences for you in Rio de Janeiro this week:

1. **Samba Night at Lapa**
   - **Description:** Dance the night away at a traditional samba club in the Lapa arches.
   - **Date:** October 20, 2026
   - **Slots Available:** 11

2. **Christ the Redeemer Experience** (Premium)
   - **Description:** Take a guided trip to one of the New Sev

In [15]:
# Long-term memories stored for Bob across tickets
from agentic.tools import ticket_tools
state = orchestrator.get_state({"configurable": {"thread_id": "demo-7"}}).values
ticket_tools.get_memories(state["customer"]["udahub_user_id"])

[{'category': 'issue_history',
  'content': 'Reported an issue with not receiving the password reset email.'},
 {'category': 'preference',
  'content': 'Enjoys music and dance experiences, especially in Rio de Janeiro'}]

In [16]:
# Explicit request for a human → immediate escalation
show(process_ticket("This is ridiculous, I want to talk to a real person now.", "demo-8", {"external_user_id": "88382b"}))

   [memory] ticket=demo-8 customer=Cathy Bloom identified=True status=open
   [classifier] other urgency=medium sentiment=frustrated account_action=False confidence=0.80
   [supervisor] route -> escalation_agent: customer asked for a human
   [escalation_agent] -> general_support (medium): customer asked for a human
   [finalize] status=escalated
   [memory] stored 0 new memories: []

Assistant: Dear Cathy,

Thank you for reaching out. I understand that you would like to speak with a real person for assistance. A specialist from our team will follow up with you shortly during our business hours (Monday to Friday, 9:00 AM to 6:00 PM). You can expect a reply via email within one business day.

If you have any other questions in the meantime, please feel free to ask.

Best regards,
CultPass Support Team 



## Interactive chat

Type `quit` to leave. The thread id is the ticket id.

In [ ]:
chat_interface(orchestrator, "1")

In [ ]:
list(orchestrator.get_state_history(
    config = {
        "configurable": {
            "thread_id": "1",
        }
    }
))[0].values["messages"]